# Hamra Coffee Shop – Data Analytics Project ☕

## 1. Project Overview

This notebook contains the complete data analytics workflow for a Lebanese-style coffee shop located in **Hamra**.  
The goal of the project is to transform raw CSV files (sales, products, customers, time) into actionable business insights and a clean dataset ready for visualization in **Tableau**.

**Main objectives:**
- Load and inspect the raw data from CSV files.
- Clean and prepare the data using Python (pandas, numpy).
- Create new features and KPIs (e.g., total sales per transaction, sales by category, peak hours).
- Perform exploratory data analysis (EDA) to understand sales patterns.
- Export a final, clean dataset for Tableau dashboarding.

**Datasets used:**
- `Products.csv` – menu items (coffee, manakish, salads, desserts, etc.)
- `Customers.csv` – customer demographics.
- `Sales.csv` – transactions for Hamra branch during 2024.
- `Time.csv` – full date-time dimension for the year 2024.

## 2. Importing Libraries & Loading Data

In [1]:
import pandas as pd
import numpy as np

### 2.1 Data Loading & Initial Preview

In [2]:
# Loading the four main datasets into pandas DataFrames

df_products = pd.read_csv("Products.csv")      # Load product catalog (menu items)
df_customers = pd.read_csv("Customers.csv")    # Load customer demographic information
df_sales = pd.read_csv("Sales.csv")            # Load sales transactions for the Hamra branch
df_time = pd.read_csv("Time.csv")              # Load full 2024 time-dimension table

# Display first rows of each dataset to verify structure and correct loading
display(df_products.head())     # Preview products
display(df_customers.head())    # Preview customers
display(df_sales.head())        # Preview sales transactions
display(df_time.head())         # Preview time-dimension table

,Product_ID,Product_Name,Category,Unit_Price,Cost
0,101,Espresso,Coffee,2.5,0.5
1,102,Turkish Coffee,Coffee,2.0,0.4
2,103,Blonde Coffee,Coffee,2.2,0.5
3,104,Americano,Coffee,3.0,0.6
4,105,Café Latte,Coffee,4.0,0.8


,Customer_ID,Gender,Age,City
0,C001,F,18,Zahle
1,C002,F,37,Zahle
2,C003,F,51,Batroun
3,C004,M,23,Tyre
4,C005,M,33,Beirut


,Order_ID,Date,Hour,Product_ID,Quantity,Customer_ID,Branch
0,3764,2024-04-12,11,502,1,C150,Hamra
1,3475,2024-12-14,10,407,1,C138,Hamra
2,3374,2024-09-27,9,103,3,C135,Hamra
3,2231,2024-04-16,11,603,1,C089,Hamra
4,353,2024-03-12,20,503,1,C015,Hamra


,Time_ID,Date,Day,Month,Year,Weekday,Hour
0,1,2024-01-01,1,1,2024,Monday,7
1,2,2024-01-01,1,1,2024,Monday,8
2,3,2024-01-01,1,1,2024,Monday,9
3,4,2024-01-01,1,1,2024,Monday,10
4,5,2024-01-01,1,1,2024,Monday,11


### 2.2 Data Validation

In [3]:
# Checking the structure, data types, and non-null counts for each dataset

print("=== Products Dataset ===")
df_products.info()     # Validate dtypes, non-null counts, memory usage

print("\n=== Customers Dataset ===")
df_customers.info()    # Ensure customers loaded correctly

print("\n=== Sales Dataset ===")
df_sales.info()        # Important: checks Date dtype, Quantity dtype, and missing values

print("\n=== Time Dataset ===")
df_time.info()         # Check date and time structure for merging later

=== Products Dataset ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53 entries, 0 to 52
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Product_ID    53 non-null     int64  
 1   Product_Name  53 non-null     object 
 2   Category      53 non-null     object 
 3   Unit_Price    53 non-null     float64
 4   Cost          53 non-null     float64
dtypes: float64(2), int64(1), object(2)
memory usage: 2.2+ KB

=== Customers Dataset ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Customer_ID  200 non-null    object
 1   Gender       200 non-null    object
 2   Age          200 non-null    int64 
 3   City         200 non-null    object
dtypes: int64(1), object(3)
memory usage: 6.4+ KB

=== Sales Dataset ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entri

The `.info()` inspection confirms that all datasets were loaded without missing values.  
Numeric fields such as prices, quantities, and IDs are correctly recognized as integers or floats.

The only required correction is the `Date` column in both the **sales** and **time** datasets, which is currently stored as a string (`object`).  
This needs to be converted to `datetime` format to enable time-based analysis, merging, and proper sorting.

## 3. Data Cleaning – Date Conversion

### 3.1 Converting Date columns from string (object) to datetime

In [4]:
# Convert Date in sales dataset
df_sales["Date"] = pd.to_datetime(df_sales["Date"])   # enables time-based grouping and filtering

# Convert Date in time-dimension dataset
df_time["Date"] = pd.to_datetime(df_time["Date"])     # ensures consistent type for merging later

# Quick check: confirm new dtypes
print("Sales Date dtype:", df_sales["Date"].dtype)
print("Time Date dtype:", df_time["Date"].dtype)

Sales Date dtype: datetime64[ns]
Time Date dtype: datetime64[ns]


The `Date` column in both `df_sales` and `df_time` has been converted from string (`object`) to `datetime64[ns]`. 

This is necessary to:
- perform time-based analysis (daily, monthly, weekday trends),
- merge sales data with the time-dimension table,
- sort and filter correctly by date.

### 3.2 Duplicate Check

In [5]:
# Count duplicate rows in each dataset
dup_products = df_products.duplicated().sum()      # Product catalog should have unique rows
dup_customers = df_customers.duplicated().sum()    # Each customer should be unique
dup_sales = df_sales.duplicated().sum()            # No duplicate sales transactions expected
dup_time = df_time.duplicated().sum()              # Time dimension should have unique timestamps

# Print results
print("Duplicate rows in Products:", dup_products)
print("Duplicate rows in Customers:", dup_customers)
print("Duplicate rows in Sales:", dup_sales)
print("Duplicate rows in Time:", dup_time)

Duplicate rows in Products: 0
Duplicate rows in Customers: 0
Duplicate rows in Sales: 0
Duplicate rows in Time: 0


No duplicate rows were found in any of the four datasets.  
This confirms that all product entries, customer records, sales transactions, and time-dimension rows are unique.  
With the data validated and cleaned, we can proceed to merging the datasets for analysis.

## 4. Data Preparation – Merging Datasets

### 4.1 Merge sales data with product information using Product_ID

In [6]:
df_merged = df_sales.merge(df_products, on="Product_ID", how="left")

# Preview the result
df_merged.head()

,Order_ID,Date,Hour,Product_ID,Quantity,Customer_ID,Branch,Product_Name,Category,Unit_Price,Cost
0,3764,2024-04-12,11,502,1,C150,Hamra,Chicken Sub Sandwich,Sandwiches,7.0,3.0
1,3475,2024-12-14,10,407,1,C138,Hamra,Cheese Rolls,Appetizers,4.5,1.0
2,3374,2024-09-27,9,103,3,C135,Hamra,Blonde Coffee,Coffee,2.2,0.5
3,2231,2024-04-16,11,603,1,C089,Hamra,Fattoush,Salads,5.5,2.0
4,353,2024-03-12,20,503,1,C015,Hamra,BBQ Chicken Sandwich,Sandwiches,7.5,3.2


### 4.2 Merge with customer information using Customer_ID

In [7]:
df_merged = df_merged.merge(df_customers, on="Customer_ID", how="left")

# Preview
df_merged.head()

,Order_ID,Date,Hour,Product_ID,Quantity,Customer_ID,Branch,Product_Name,Category,Unit_Price,Cost,Gender,Age,City
0,3764,2024-04-12,11,502,1,C150,Hamra,Chicken Sub Sandwich,Sandwiches,7.0,3.0,M,46,Zahle
1,3475,2024-12-14,10,407,1,C138,Hamra,Cheese Rolls,Appetizers,4.5,1.0,F,55,Tyre
2,3374,2024-09-27,9,103,3,C135,Hamra,Blonde Coffee,Coffee,2.2,0.5,F,20,Byblos
3,2231,2024-04-16,11,603,1,C089,Hamra,Fattoush,Salads,5.5,2.0,F,21,Saida
4,353,2024-03-12,20,503,1,C015,Hamra,BBQ Chicken Sandwich,Sandwiches,7.5,3.2,F,52,Zahle


### 4.3 Merge with time-dimension table using Date and Hour

In [8]:
df_merged = df_merged.merge(df_time, on=["Date", "Hour"], how="left")

# Preview
df_merged.head()

,Order_ID,Date,Hour,Product_ID,Quantity,Customer_ID,Branch,Product_Name,Category,Unit_Price,Cost,Gender,Age,City,Time_ID,Day,Month,Year,Weekday
0,3764,2024-04-12,11,502,1,C150,Hamra,Chicken Sub Sandwich,Sandwiches,7.0,3.0,M,46,Zahle,1433,12,4,2024,Friday
1,3475,2024-12-14,10,407,1,C138,Hamra,Cheese Rolls,Appetizers,4.5,1.0,F,55,Tyre,4876,14,12,2024,Saturday
2,3374,2024-09-27,9,103,3,C135,Hamra,Blonde Coffee,Coffee,2.2,0.5,F,20,Byblos,3783,27,9,2024,Friday
3,2231,2024-04-16,11,603,1,C089,Hamra,Fattoush,Salads,5.5,2.0,F,21,Saida,1489,16,4,2024,Tuesday
4,353,2024-03-12,20,503,1,C015,Hamra,BBQ Chicken Sandwich,Sandwiches,7.5,3.2,F,52,Zahle,1008,12,3,2024,Tuesday


### 4.4 Validate row count after merges

In [9]:
original_rows = len(df_sales)
merged_rows = len(df_merged)

print("Original Sales Rows:", original_rows)
print("Merged Dataset Rows:", merged_rows)


Original Sales Rows: 5000
Merged Dataset Rows: 5000


The merged dataset retains the original 5,000 sales records, confirming that no rows were lost or duplicated during the joins.

### 4.5 Creating Total_Sale and Profit columns

In [10]:
df_merged["Total_Sale"] = df_merged["Unit_Price"] * df_merged["Quantity"]   # Revenue per transaction
df_merged["Profit"] = (df_merged["Unit_Price"] - df_merged["Cost"]) * df_merged["Quantity"]

# Preview
df_merged[["Product_Name", "Quantity", "Unit_Price", "Cost", "Total_Sale", "Profit"]].head()

,Product_Name,Quantity,Unit_Price,Cost,Total_Sale,Profit
0,Chicken Sub Sandwich,1,7.0,3.0,7.0,4.0
1,Cheese Rolls,1,4.5,1.0,4.5,3.5
2,Blonde Coffee,3,2.2,0.5,6.6,5.1
3,Fattoush,1,5.5,2.0,5.5,3.5
4,BBQ Chicken Sandwich,1,7.5,3.2,7.5,4.3


Two key metrics were added:
- **Total_Sale**: revenue for each transaction (Unit_Price × Quantity)
- **Profit**: profit per transaction ((Unit_Price - Cost) × Quantity)

These fields will be used extensively in the exploratory analysis and Tableau dashboard.

## 5. Exploratory Data Analysis (EDA)

### 5.1 Top-Selling Products (by Quantity)

In [11]:
top_products = (
    df_merged.groupby("Product_Name")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_products

Product_Name
Lemonade               172
Green Tea              166
Carrot Juice           159
Steak Sandwich         157
Lemonade Mint          155
Nutella Crepe          155
Chocolate Milkshake    153
Croissant              150
Kaak with Cheese       149
Keshek Manoushe        148
Name: Quantity, dtype: int64

- **Lemonade (172 orders)** is the most frequently purchased item, reflecting strong demand for refreshing cold drinks.
- **Green Tea (166 orders)** and **Carrot Juice (159 orders)** also perform exceptionally well, indicating that lighter, health-oriented beverages are popular choices.
- Among food items, the **Steak Sandwich (157 orders)** stands out as the top savory best-seller.
- Desserts such as **Nutella Crepe (155 orders)** and drinks like **Chocolate Milkshake (153 orders)** show strong performance in the sweet category.
- Classic bakery choices like the **Croissant (150 orders)** and regional items such as **Kaak with Cheese (149 orders)** and **Keshek Manoushe (148 orders)** remain consistently popular.

### 5.2 Highest Revenue Products

In [12]:
top_revenue_products = (
    df_merged.groupby("Product_Name")["Total_Sale"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_revenue_products

Product_Name
Steak Sandwich          1413.0
Chicken Caesar Salad    1027.5
Greek Salad              949.0
Halloumi Salad           897.6
BBQ Chicken Sandwich     892.5
Banane Avocat            864.5
Nutella Crepe            852.5
Mango Cocktail           846.0
Chocolate Milkshake      841.5
Ashta W Banane           819.0
Name: Total_Sale, dtype: float64

- The **Steak Sandwich (1,413 USD)** is the top revenue generator.  
  Although it was not the most frequently purchased item, its higher price makes it the café's most financially valuable product.

- **Chicken Caesar Salad (1,027.5 USD)**, **Greek Salad (949 USD)**, and **Halloumi Salad (897.6 USD)** show that salads contribute significantly to revenue, likely due to higher margins and strong demand among health-conscious customers.

- **BBQ Chicken Sandwich (892.5 USD)** also demonstrates strong financial performance, reinforcing the popularity of premium sandwiches.

- Sweet and blended items such as **Banane Avocat (864.5 USD)**, **Nutella Crepe (852.5 USD)**, and **Chocolate Milkshake (841.5 USD)** rank highly, indicating consistent customer interest in desserts and specialty beverages.

- The **Mango Cocktail (846 USD)** and **Ashta W Banane (819 USD)** also highlight strong revenue from the cocktails category.

### 5.3 Most Profitable Products

In [13]:
top_profit_products = (
    df_merged.groupby("Product_Name")["Profit"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_profit_products

Product_Name
Steak Sandwich          785.0
Chicken Caesar Salad    589.1
Nutella Crepe           573.5
Chocolate Milkshake     566.1
Greek Salad             540.2
Mango Cocktail          535.8
Banane Avocat           532.0
Strawberry Cocktail     532.0
Ashta W Banane          526.5
Halloumi Salad          514.8
Name: Profit, dtype: float64

- The **Steak Sandwich (785 USD profit)** is the café's most profitable item.  
  It ranks #1 in both revenue and profit, making it a core high-value product for the business.

- **Chicken Caesar Salad (589 USD)** also performs strongly, indicating salads are not only popular but also high-margin items.

- Dessert and beverage items contribute substantially to overall profit:  
  - **Nutella Crepe (573.5 USD)**  
  - **Chocolate Milkshake (566.1 USD)**  
These items likely maintain strong profit due to relatively low cost of preparation compared to their selling price.

- **Greek Salad (540.2 USD)** and **Mango Cocktail (535.8 USD)** further confirm that healthier or specialty items can be both revenue and profit drivers.

- **Banane Avocat**, **Strawberry Cocktail**, and **Ashta W Banane** also appear, reinforcing the importance of cocktails and specialty drinks as profitable categories.

### 5.4 Hourly Sales Distribution

In [14]:
hourly_orders = (
    df_merged.groupby("Hour")["Order_ID"]
    .nunique()    # unique orders in each hour
    .sort_index()
)

hourly_orders

Hour
7     340
8     373
9     345
10    343
11    339
12    369
13    397
14    344
15    343
16    395
17    336
18    360
19    376
20    331
Name: Order_ID, dtype: int64

**Key insights:**
- **Peak hours are 1 PM (397 orders) and 4 PM (395 orders).**  
  These times likely reflect lunch rush and afternoon breaks.

- The café maintains **steady morning activity (7–11 AM)**, driven by coffee, bakery items, and manakish orders.

- **Evening hours (6–7 PM)** remain active with dessert and drink orders.

- The **slowest hour is 8 PM (331 orders)** — still relatively strong, indicating consistent demand throughout the day.

**Business implications:**
- Staffing levels should be highest around **1 PM and 4 PM**.
- Inventory planning for high-volume hours can reduce waste and shortages.
- Promos during slow hours (e.g., 8 PM) could increase foot traffic.
- Menu planning should consider time-specific preferences (morning coffee peaks, afternoon snacks, evening desserts).

### 5.5 Weekday Sales Analysis

In [15]:
weekday_orders = (
    df_merged.groupby("Weekday")["Order_ID"]
    .nunique()
    .reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])  #  sort per day order
)

weekday_orders

Weekday
Monday       716
Tuesday      716
Wednesday    693
Thursday     681
Friday       725
Saturday     735
Sunday       725
Name: Order_ID, dtype: int64

- **Saturday is the busiest day of the week (735 orders).**  
  This is typical for cafés and small restaurants, where weekend outings drive higher traffic.

- **Friday and Sunday also show strong performance (725 orders each),** reinforcing the importance of the weekend period.

- **Monday and Tuesday maintain solid activity (716 orders each),** showing a stable start to the week.

- **Wednesday and Thursday are the slowest days**, although the difference is relatively small.  
  These days could benefit from targeted promotions or special offers to boost mid-week traffic.

### 5.6 Monthly Sales Trends

In [16]:
monthly_orders = (
    df_merged.groupby("Month")["Order_ID"]
    .nunique()
    .sort_index()     # keep months 1→12
)

monthly_orders

Month
1     418
2     401
3     398
4     433
5     449
6     434
7     377
8     427
9     398
10    422
11    404
12    430
Name: Order_ID, dtype: int64

**Key insights:**

- **May is the peak month (449 orders)**, showing the highest customer activity of the year.
- **June (434) and April (433)** also show strong performance, suggesting late spring and early summer are high-demand periods.
- **August (427) and December (430)** remain solid, indicating steady traffic during vacation and holiday periods.
- **July is the slowest month (377 orders)**, which may reflect seasonal slowdowns, heat, or travel habits.
- Despite variations, order volume remains relatively balanced throughout the year, suggesting consistent customer loyalty.

**Business implications:**
- Inventory planning should consider the spikes in May, April, and June.
- July could benefit from targeted promotions to increase customer traffic.
- Seasonal menu adjustments may help align product offerings with monthly trends.

### 5.7 Customer Segmentation

In [17]:
# Gender distribution

gender_counts = df_customers["Gender"].value_counts()
gender_counts

Gender
F    105
M     95
Name: count, dtype: int64

The gender split is relatively balanced, with a slight majority of female customers (52.5%). This balance helps ensure that menu and marketing strategies can target both groups effectively.

In [20]:
# Age distribution

age_stats = df_merged["Age"].describe()
age_stats

count    5000.000000
mean       37.228000
std        12.500851
min        18.000000
25%        26.000000
50%        37.000000
75%        48.000000
max        60.000000
Name: Age, dtype: float64

The age statistics for customers are:

- **Mean age:** 37.2 years  
- **Youngest:** 18  
- **Oldest:** 60  
- **Most customers fall between 26–48 years old**

This shows that the café attracts a mature and diverse age range, with a strong middle-age customer base.

### 5.8 Average Order Value (AOV) by Gender

In [32]:
# Average order value per gender
aov_gender = (
    df_merged.groupby("Gender")["Total_Sale"]
    .mean()
    .round(2)
)
aov_gender

Gender
F    6.42
M    6.36
Name: Total_Sale, dtype: float64

Both genders spend almost the same amount per order.  
Female customers have a slightly higher AOV (+0.06), but the difference is not significant.

### 5.9 Most Popular Categories by Gender

In [24]:
popular_category_gender = (
    df_merged.groupby(["Gender", "Category"])["Quantity"]
    .sum()
    .sort_values(ascending=False)
)
popular_category_gender.head(10)

Gender  Category   
F       Coffee         572
M       Coffee         491
F       Sandwiches     397
M       Sandwiches     367
F       Salads         348
        Tea            346
        Desserts       338
M       Tea            329
F       Cold Drinks    317
M       Desserts       316
Name: Quantity, dtype: int64

✔ *Coffee is the most popular category across both genders*  
✔ Female customers buy slightly more from almost every category  
✔ Salads and Tea have stronger popularity among female customers  
✔ Cold Drinks are more popular among male customers compared to desserts

### 5.10 Age Group Creation

In [25]:
df_merged["Age_Group"] = pd.cut(
    df_merged["Age"],
    bins=[15, 25, 35, 50, 80],
    labels=["16–25", "26–35", "36–50", "50+"],
    include_lowest=True
)

df_merged[["Age", "Age_Group"]].head()

,Age,Age_Group
0,46,36–50
1,55,50+
2,20,16–25
3,21,16–25
4,52,50+


### 5.11 Popular Categories by Age Group

In [31]:
popular_category_age = (
    df_merged.groupby(["Age_Group", "Category"])["Quantity"]
    .sum()
    .sort_values(ascending=False)
)
popular_category_age.head(10)

C:\Users\wael\AppData\Local\Temp\ipykernel_18676\2138136336.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_merged.groupby(["Age_Group", "Category"])["Quantity"]


Age_Group  Category   
36–50      Coffee         374
26–35      Coffee         284
36–50      Sandwiches     276
           Desserts       254
           Salads         254
16–25      Coffee         239
36–50      Tea            238
           Cold Drinks    221
           Appetizers     205
           Manakish       190
Name: Quantity, dtype: int64

**36–50 age group (largest impact)**  
- Coffee – 374  
- Sandwiches – 276  
- Desserts – 254  
- Salads – 254  
- Tea – 238  
- Cold Drinks – 221  
- Appetizers – 205  

This group buys the **most items overall**, making them the café’s most valuable demographic.

**26–35 group**
- Coffee – 284  

This group is highly coffee-driven, likely due to work routines and takeaway orders.

**16–25 group**
- Coffee – 239  

Younger customers also prefer coffee but order fewer food items.

- **Coffee dominates across all genders and age groups**, making it the café’s core product category.
- Customers aged **36–50** are the most active and generate the highest item volume.
- Female customers purchase slightly more and across more categories.
- Age drives category preferences:  
  - Younger groups: mostly coffee  
  - Mid-age groups: balanced mix (coffee, sandwiches, salads, desserts)  
  - Older groups: tea and salads become more important

### 5.12 Order-Level Metrics (AOV & Items per Order)

In [38]:
# Total revenue across all orders
total_revenue = df_merged["Total_Sale"].sum()

# Number of unique orders
num_orders = df_merged["Order_ID"].nunique()

# AOV calculation
AOV = round(total_revenue / num_orders, 2)
print('AOV =', AOV)

AOV = 6.41


In [39]:
items_per_order = (
    df_merged.groupby("Order_ID")["Quantity"]
    .sum()
    .mean()
    .round(2)
)

print('items_per_order =', items_per_order)

items_per_order = 1.41


In [40]:
order_item_dist = (
    df_merged.groupby("Order_ID")["Quantity"]
    .sum()
    .value_counts()
    .sort_index()
)

print('order_item_dist =', order_item_dist)

order_item_dist = Quantity
1    3475
2    1006
3     509
4       1
Name: count, dtype: int64


**AOV = 6.41 USD**

Customers spend an average of **$6.41 per visit**.  
For a café–restaurant in Hamra, this is a highly realistic value, reflecting typical purchases such as:
- a coffee + small snack,
- a juice,
- a sandwich,
- a dessert item.

AOV is stable and suggests consistent spending habits.

**Average Items per Order = 1.41**

Customers buy approximately **1 to 2 items per visit**.  
This indicates:
- many customers stop for a single item (mostly coffee, tea, or juice),
- a smaller but meaningful portion buy full meals (sandwiches + drinks).

This ratio is typical for cafés with mixed quick-stop and dine-in traffic.

- **69.5%** of orders are **1-item transactions**, showing strong quick-consumption behavior (coffee, manakish, juices).
- **20.1%** of orders contain **2 items**, likely a drink + food combo.
- **10.2%** of orders contain **3 items**, representing small groups or dine-in customers.
- Only **1 order** contained 4 items, which is expected for a single-customer ordering structure.


### 5.13 Category-Level Analysis

#### 5.13.1 Total quantity sold by category

In [42]:
category_quantity = (
    df_merged.groupby("Category")["Quantity"]
    .sum()
    .sort_values(ascending=False)
)

category_quantity

Category
Coffee                1063
Sandwiches             764
Tea                    675
Desserts               654
Salads                 654
Cold Drinks            602
Appetizers             590
Manakish               519
Cocktails              407
Shakes                 399
Bakery                 299
Hot Drinks             275
Cocktails Specials     117
Name: Quantity, dtype: int64

- **Coffee** is the top-selling category (1063 items), confirming it as the café’s primary driver of foot traffic.
- **Sandwiches**, **Tea**, **Desserts**, and **Salads** also show strong volume, reflecting a balanced mix of food and beverage consumption.
- Lower-volume categories such as **Bakery**, **Hot Drinks**, and especially **Cocktails Specials** can be considered niche offerings or seasonal items.

#### 5.13.2 Total revenue by category

In [47]:
category_revenue = (
    df_merged.groupby("Category")["Total_Sale"]
    .sum()
    .sort_values(ascending=False)
)

category_revenue

Category
Sandwiches            5343.0
Salads                4188.6
Coffee                3510.6
Desserts              3423.9
Cocktails             2508.5
Shakes                2254.0
Manakish              2192.0
Appetizers            2127.8
Cold Drinks           2021.0
Tea                   1537.8
Hot Drinks            1074.0
Bakery                 971.5
Cocktails Specials     819.0
Name: Total_Sale, dtype: float64

- **Sandwiches** generate the highest revenue, despite not being the highest in quantity — showing higher price points and meal-oriented consumption.
- **Salads** rank second in revenue, showing strong interest in fresh and healthy offerings.
- **Coffee** generates substantial revenue due to its massive volume.
- **Desserts**, **Cocktails**, and **Shakes** also contribute meaningfully to sales.

#### 5.13.3 Total profit by category

In [45]:
category_profit = (
    df_merged.groupby("Category")["Profit"]
    .sum()
    .sort_values(ascending=False)
)

category_profit

Category
Sandwiches            3150.7
Coffee                2794.4
Salads                2480.6
Desserts              2270.8
Manakish              1783.7
Cocktails             1599.8
Appetizers            1579.0
Shakes                1488.2
Cold Drinks           1453.4
Tea                   1212.0
Hot Drinks             812.0
Bakery                 732.3
Cocktails Specials     526.5
Name: Profit, dtype: float64

- **Sandwiches** lead in total profit, making them the café’s most financially important category.
- **Coffee**, despite a low unit price, generates very high profit due to volume and excellent margins.
- **Salads** and **Desserts** perform well in both revenue and profit, showing strong customer demand and good cost control.
- **Manakish**, **Appetizers**, **Cocktails**, and **Shakes** form a strong secondary profit group.
- **Cocktails Specials** generate low profit due to low volume.

#### 5.13.4 Average unit price per category

In [46]:
category_avg_price = (
    df_products.groupby("Category")["Unit_Price"]
    .mean()
    .round(2)
    .sort_values(ascending=False)
)

category_avg_price

Category
Cocktails Specials    7.00
Sandwiches            6.92
Salads                6.36
Cocktails             6.17
Shakes                5.67
Desserts              5.26
Manakish              4.25
Hot Drinks            3.90
Appetizers            3.64
Cold Drinks           3.38
Coffee                3.31
Bakery                3.25
Tea                   2.28
Name: Unit_Price, dtype: float64

- **High-price categories**:  
  Cocktails Specials, Sandwiches, Salads, and Cocktails.  
  These reflect premium, full-meal, or specialty items.

- **Low-price categories**:  
  Tea, Coffee, Bakery, Cold Drinks, Hot Drinks — essential for daily customers and high traffic.

- **Coffee dominates in volume**, making it the café’s primary traffic driver.
- **Sandwiches lead in both revenue and profit**, positioning them as strategic menu anchors.
- **Salads and Desserts** show strong all-around performance — appealing, profitable categories.
- **Cocktails and Shakes** deliver high revenue and good profit, supporting a premium drink strategy.
- **Manakish and Appetizers** remain reliable supportive categories.
- **Cocktails Specials** are niche premium products but very low in volume.

## 6. Conclusion & Recommendation

### 6.1 Conclusion

The analysis of the café’s 2024 performance reveals a strong and balanced business with consistent customer traffic, reliable revenue streams, and a healthy mix of product categories.

Coffee acts as the primary volume driver, while sandwiches generate the highest revenue and profit, making them the financial backbone of the menu. Salads and desserts also demonstrate strong performance, appealing to multiple customer segments. 

Customer behavior shows a mature and loyal base, with an average age of 37 and a nearly even gender split.

Most customers purchase one item per visit, but a significant portion order two or more items, providing opportunities to increase Average Order Value through strategic upselling. 

Seasonal and time-based trends indicate clear peak periods during lunch and afternoon hours, as well as higher activity during weekends and certain months like May, June, and December.

Overall, the café’s performance is stable, predictable, and offers multiple levers for improving revenue and customer engagement through data-driven decisions.


### 6.2. Recommendations

**1. Increase Average Order Value (AOV)**
- Introduce combo deals (e.g., Coffee + Croissant, Sandwich + Drink).  
- Add automatic POS prompts for upselling desserts or sides.  
- Highlight meal bundles on the menu and in-store displays.

**2. Strengthen High-Value Categories**
- Feature sandwiches, salads, and desserts more prominently—they deliver strong profit and revenue.  
- Use coffee as a lead-in product, with targeted promotions encouraging add-ons.

**3. Target Customer Segments Effectively**
- Ages 26–50 are the most active and profitable—design promotions and loyalty programs around them.  
- Younger customers (16–25) respond strongly to coffee and cold drinks—promote these via social media.  
- Older customers prefer tea and lighter meals—optimize offerings for this group.

**4. Optimize Operations Based on Time Trends**
- Increase staffing and preparation during peak hours (1 PM and 4 PM).  
- Introduce mid-week promotions to boost Wednesday and Thursday traffic.  
- Launch seasonal campaigns during slower months such as July.

**5. Improve Menu and Inventory Strategy**
- Maintain strong inventory for high-volume categories: Coffee, Sandwiches, Salads, Desserts.  
- Assess low-volume categories (e.g., Cocktails Specials) for seasonal or limited-time positioning.  
- Redesign menu layout to emphasize high-margin items and healthy best-sellers.

**6. Enhance Marketing and Customer Engagement**
- Use demographic insights to run targeted digital ads.  
- Promote premium categories (Cocktails, Shakes, Salads) during evening hours.  
- Build loyalty incentives around repeat visits and bundled purchases.

Together, these recommendations provide a clear roadmap to increase revenue, improve customer experience, and optimize daily operations using data-driven decision-making.

In [48]:
# Export final dataset for Tableau
df_merged.to_csv("Final_Cafe_Dataset_For_Tableau.csv", index=False)

print("CSV exported successfully!")

CSV exported successfully!


Find Tableau Dashboards Link here:

https://public.tableau.com/views/HamraCoffeeShop2/Dashboard1?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link

https://public.tableau.com/views/HamraCoffeeShop2/Dashboard2?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link